# Predictor Notebook – Müllerkennung

Dieses Notebook ist getrennt vom Training und dient nur dazu, mit dem bereits trainierten Modell neue Bilder vorherzusagen.

## Ziel
Du lädst ein beliebiges Bild in den Projektordner oder gibst einen Bildpfad an, und das Modell sagt vorher, ob das Bild zu einer der folgenden Klassen gehört:

- `cardboard`
- `glass`
- `plastic`

## Voraussetzung
Bevor dieses Notebook funktioniert, muss das trainierte Modell bereits vorhanden sein:

- `trash_classifier_3classes.keras`

Dieses Modell wurde im Training-Notebook gespeichert.

## Ablauf
1. Projektordner finden  
2. Gespeichertes Modell laden  
3. Klassen definieren  
4. Einzelnes Bild laden und vorbereiten  
5. Vorhersage anzeigen  
6. Optional mehrere Bilder nacheinander testen


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing import image


## 1. Projektordner automatisch finden

Hier wird nur sichergestellt, dass das Notebook im richtigen Projektordner arbeitet.  
Es werden dabei keine Dateien geändert.


In [ ]:
REQUIRED_FILE = "trash_classifier_3classes.keras"

def find_project_root(start_path, required_file=REQUIRED_FILE):
    current = start_path
    while True:
        if required_file in os.listdir(current):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            return None
        current = parent

print("Aktueller Arbeitsordner:", os.getcwd())

project_root = find_project_root(os.getcwd())

if project_root is None:
    raise FileNotFoundError(f"'{REQUIRED_FILE}' konnte nicht gefunden werden.")

if os.getcwd() != project_root:
    os.chdir(project_root)
    print("Arbeitsverzeichnis gesetzt auf:", project_root)
else:
    print("Arbeitsverzeichnis war schon korrekt.")

print("Inhalt des Projektordners:")
print(os.listdir())


## 2. Modell laden

Hier wird das bereits trainierte Modell geladen.  
Es wird nicht neu trainiert, sondern nur wiederverwendet.


In [ ]:
MODEL_PATH = "trash_classifier_3classes.keras"
model = keras.models.load_model(MODEL_PATH)

print("Modell erfolgreich geladen.")
model.summary()


## 3. Klassen und Bildgröße festlegen

Diese Werte müssen zur Training-Datei passen.


In [ ]:
TARGET_CLASSES = ["cardboard", "glass", "plastic"]
IMG_SIZE = (224, 224)

print("Klassen:", TARGET_CLASSES)
print("Bildgröße:", IMG_SIZE)


## 4. Vorhersagefunktion für ein einzelnes Bild

Diese Funktion:
- lädt ein Bild
- passt es auf `224x224` an
- gibt die vorhergesagte Klasse aus
- zeigt zusätzlich die Wahrscheinlichkeiten pro Klasse


In [ ]:
def predict_image(img_path):
    if not os.path.exists(img_path):
        raise FileNotFoundError(f"Bild nicht gefunden: {img_path}")

    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array, verbose=0)[0]
    pred_index = int(np.argmax(preds))
    pred_class = TARGET_CLASSES[pred_index]

    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.title(f"Prediction: {pred_class}")
    plt.axis("off")
    plt.show()

    print("Vorhergesagte Klasse:", pred_class)
    print("\nWahrscheinlichkeiten:")
    for class_name, prob in zip(TARGET_CLASSES, preds):
        print(f"  {class_name}: {prob:.4f}")

    return pred_class, preds


## 5. Ein einzelnes Bild testen

### So benutzt du die Funktion
Lege ein Bild in deinen Projektordner oder in einen Unterordner und passe den Pfad unten an.

Beispiel:
- `testbild.jpg`
- `bilder/testbild.jpg`


In [ ]:
# HIER den Bildpfad anpassen
img_path = "testbild.jpg"

# Beispielaufruf
# predict_image(img_path)


Wenn du den Pfad oben angepasst hast, kannst du diese Zelle ausführen:


In [ ]:
predict_image(img_path)


## 6. Mehrere Bilder nacheinander testen

Wenn du mehrere Bilder in einem Ordner hast, kannst du sie automatisch durchgehen lassen.


In [ ]:
def predict_images_from_folder(folder_path):
    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f"Ordner nicht gefunden: {folder_path}")

    valid_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    files = [f for f in os.listdir(folder_path) if f.lower().endswith(valid_extensions)]

    if not files:
        print("Keine unterstützten Bilddateien gefunden.")
        return

    print(f"{len(files)} Bild(er) gefunden in: {folder_path}\n")

    for file in files:
        full_path = os.path.join(folder_path, file)
        print("=" * 60)
        print("Datei:", full_path)
        predict_image(full_path)


### Beispiel für einen ganzen Ordner
Lege z. B. mehrere Bilder in einen Ordner `predict_images` und führe dann die Zelle unten aus.


In [ ]:
# Beispiel:
# predict_images_from_folder("predict_images")


## 7. Optional: Ein Testbild aus dem vorhandenen Datensatz prüfen

Wenn du schnell etwas ausprobieren willst, kannst du auch ein Bild aus dem vorhandenen Datensatz verwenden.


In [ ]:
example_path = os.path.join("dataset-split", "test", "glass")

if os.path.isdir(example_path):
    files = os.listdir(example_path)
    if files:
        sample_image = os.path.join(example_path, files[0])
        print("Beispielbild:", sample_image)
    else:
        print("Keine Dateien im Beispielordner gefunden.")
else:
    print("Beispielordner nicht gefunden.")


In [ ]:
# Optionaler Test mit vorhandenem Bild aus dem Datensatz
# predict_image(sample_image)


## Kurz erklärt: Was macht dieses Notebook?

Dieses Notebook verwendet das bereits trainierte Modell wie einen Predictor.

Das bedeutet:
- Es wird nicht erneut trainiert
- Es lädt nur das gespeicherte Modell
- Danach kann es auf neue Bilder angewendet werden

So kann man das Modell praktisch nutzen, also genau das machen, wofür es trainiert wurde.
